# Estrategias de Chunking Avanzadas

En este notebook exploraremos diferentes estrategias de chunking más allá del simple splitting por caracteres fijos, que es lo que normalmente se muestra en los tutorials básicos.

## ¿Por qué el chunking importa en RAG?

El chunking determina cómo dividimos nuestros documentos en piezas manejables para el embedding y recuperación. Un mal chunking puede llevar a:
- Pérdida de contexto importante
- Cortes en medio de oraciones o conceptos
- Chunks demasiado grandes o pequeños
- Dificultad para recuperar información relevante

En producción, necesitamos estrategias que entiendan la estructura semántica del documento.

## 1. Chunking por Caracteres Fijos (Lo que NO debemos hacer)

Este es el enfoque típico de los tutorials de 10 minutos, pero tiene graves limitaciones.

In [1]:
import re

def fixed_size_chunking(text, chunk_size=500, overlap=50):
    """Chunking por tamaño fijo de caracteres"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    return chunks

# Ejemplo
sample_text = """En un lugar de la Mancha, de cuyo nombre no quiero acordarme,
no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,
adarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero,
salpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes,
algún palomino de añadidura los domingos, consumían las tres partes de su hacienda."""

chunks = fixed_size_chunking(sample_text, chunk_size=100, overlap=20)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {repr(chunk)}")
    print()


Chunk 1: 'En un lugar de la Mancha, de cuyo nombre no quiero acordarme,\nno ha mucho tiempo que vivía un hidalg'

Chunk 2: ' que vivía un hidalgo de los de lanza en astillero,\nadarga antigua, rocín flaco y galgo corredor. Un'

Chunk 3: 'y galgo corredor. Una olla de algo más vaca que carnero,\nsalpicón las más noches, duelos y quebranto'

Chunk 4: ', duelos y quebrantos los sábados, lantejas los viernes,\nalgún palomino de añadidura los domingos, c'

Chunk 5: 'dura los domingos, consumían las tres partes de su hacienda.'



Problemas observables en el output:
- Cortes en medio de palabras
- Separa oraciones lógicamente conectadas
- No respeta la estructura narrativa

## 2. Chunking por Párrafos (Mejor, pero aún limitado)


In [2]:
def paragraph_chunking(text, max_chunk_size=500):
    """Chunking por párrafos naturales"""
    # Dividir por doble salto de línea (párrafos)
    paragraphs = re.split(r'\n\s*\n', text)
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
            
        # Si añadir este párrafo excede el tamaño máximo, empezamos un nuevo chunk
        if len(current_chunk) + len(para) > max_chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            if current_chunk:
                current_chunk += "\n\n" + para
            else:
                current_chunk = para
    
    # Añadir el último chunk si existe
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks

# Ejemplo con el mismo texto
chunks = paragraph_chunking(sample_text, max_chunk_size=200)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({(len(chunk))} chars):")
    print(repr(chunk))
    print()


Chunk 1 (380 chars):
'En un lugar de la Mancha, de cuyo nombre no quiero acordarme,\nno ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,\nadarga antigua, rocín flaco y galgo corredor. Una olla de algo más vaca que carnero,\nsalpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes,\nalgún palomino de añadidura los domingos, consumían las tres partes de su hacienda.'



Ventajas:
- Respeta límites naturales de párrafos
- Mantiene unidades semánticas más coherentes

Limitaciones:
- Los párrafos pueden variar mucho en tamaño
- Algunos documentos no tienen claro delimitado de párrafos
- No considera la jerarquía del documento (títulos, secciones)

## 3. Chunking Semántico Basado en Similitud (Enfoque de Producción)

Esta estrategia agrupa oraciones que son semánticamente similares, independientemente de su posición en el texto.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def semantic_chunking(text, model_name='all-MiniLM-L6-v2', threshold=0.7, max_chunk_size=512):
    """Chunking semántico basado en similitud de embeddings"""
    # Cargar modelo de embeddings
    model = SentenceTransformer(model_name)
    
    # Dividir texto en oraciones
    sentences = re.split(r'[.!?]+\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if len(sentences) <= 1:
        return sentences
    
    # Generar embeddings para cada oración
    embeddings = model.encode(sentences)
    
    # Calcular similitud entre oraciones consecutivas
    similarities = []
    for i in range(len(embeddings)-1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        similarities.append(sim)
    
    # Agrupar oraciones basadas en similitud
    chunks = []
    current_chunk = [sentences[0]]
    current_length = len(sentences[0])
    
    for i in range(1, len(sentences)):
        # Verificar si debemos continuar el chunk actual
        sim_with_previous = similarities[i-1] if i-1 < len(similarities) else 0
        
        # Condiciones para continuar el chunk:
        # 1. Similitud suficiente con la oración anterior
        # 2. No exceder el tamaño máximo
        if (sim_with_previous >= threshold and 
            current_length + len(sentences[i]) <= max_chunk_size):
            current_chunk.append(sentences[i])
            current_length += len(sentences[i])
        else:
            # Finalizar chunk actual y comenzar uno nuevo
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
            current_length = len(sentences[i])
    
    # Añadir último chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

# Ejemplo
chunks = semantic_chunking(sample_text, threshold=0.5)
print(f"Número de chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1} ({(len(chunk))} chars):")
    print(repr(chunk))


/home/developer/proyectos/tlc/rag-production-system/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16878.82it/s]


Número de chunks: 1

Chunk 1 (379 chars):
'En un lugar de la Mancha, de cuyo nombre no quiero acordarme,\nno ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,\nadarga antigua, rocín flaco y galgo corredor Una olla de algo más vaca que carnero,\nsalpicón las más noches, duelos y quebrantos los sábados, lantejas los viernes,\nalgún palomino de añadidura los domingos, consumían las tres partes de su hacienda.'
